In [ ]:
import shutil
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from ultralytics import YOLO

Sort StreetSurfaceVis into smoothness classes

In [ ]:
CSV_PATH = "Datasets/StreetSurfaceVis/streetSurfaceVis_v1_0.csv"
IMAGE_DIR = Path("Datasets/StreetSurfaceVis/s_original")
OUTPUT_DIR = Path("Datasets/StreetSurfaceVis/SSV_smoothness")

VAL_RATIO = 0.2
RANDOM_SEED = 42

In [ ]:
# Load csv annotation file
df = pd.read_csv(CSV_PATH)

# Smoothness class only
df["label"] = df["surface_quality"].str.replace(" ", "_")

# Use original train/test split from the CSV
train_val_df = df[df["train"] == True].copy()
test_df = df[df["train"] == False].copy()


# Create validation split from training data
# Stratified by smoothness class
val_df = (
    train_val_df
    .groupby("label", group_keys=False)
    .sample(frac=VAL_RATIO, random_state=RANDOM_SEED)
)

train_df = train_val_df.drop(val_df.index)

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

In [ ]:
# Find image file
def find_image(image_id):
    image_id = str(image_id)

    for ext in [".jpg", ".jpeg", ".png"]:
        img_path = IMAGE_DIR / f"{image_id}{ext}"
        if img_path.exists():
            return img_path

    return None

In [ ]:
# Copy images
missing_images = []

for _, row in final_df.iterrows():
    image_id = row["mapillary_image_id"]
    label = row["label"]
    split = row["split"]

    src_path = find_image(image_id)

    if src_path is None:
        missing_images.append(image_id)
        continue

    dst_dir = OUTPUT_DIR / split / label
    dst_dir.mkdir(parents=True, exist_ok=True)

    dst_path = dst_dir / src_path.name
    shutil.copy2(src_path, dst_path)

In [ ]:
# Save split annotation file
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
final_df.to_csv(OUTPUT_DIR / "split_annotations_smoothness.csv", index=False)

In [ ]:
# Print summary
print("Finished splitting dataset by smoothness class.")
print()

print("Split distribution:")
print(final_df["split"].value_counts())
print()

print("Smoothness class distribution:")
print(final_df.groupby(["split", "label"]).size())

if missing_images:
    print()
    print(f"Warning: {len(missing_images)} images were not found.")
    print("First missing images:")
    print(missing_images[:10])
else:
    print()
    print("All images were copied successfully.")

Model Training on StreetSurfaceVis

In [ ]:
DATASET_DIR = Path("Datasets/StreetSurfaceVis/SSV_smoothness").resolve()

MODEL_WEIGHTS = "src/yolo11x-cls.pt"

EPOCHS = 50

PATIENCE = 10

IMGSZ = 224
BATCH = 32
DEVICE = 0  # or "cpu"

PROJECT = Path.cwd() / f"ModelTraining/SSV_YOLO11_FFT"

PHASE_NAME = "SSV_YOLO11x_Full-fine_tuning"
TEST_NAME   = "SSV_YOLO11x_test"

In [ ]:
common_args = {
    "data": str(DATASET_DIR),
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": DEVICE,

    "patience": PATIENCE,

    "seed": RANDOM_SEED,
    "deterministic": True,

    "plots": True,
    "verbose": True,

    "project": PROJECT,
    "exist_ok": True,
}

In [ ]:
print("=" * 70)
print("Full Fine-Tuning")
print("=" * 70)

model = YOLO(MODEL_WEIGHTS)


results_phase = model.train(
    **common_args,
    epochs=EPOCHS,
    name=PHASE_NAME,
    freeze=None
)


phase_dir = Path(model.trainer.save_dir)

phase_best = phase_dir / "weights" / "best.pt"

print("/nOutput directory:")
print(phase_dir)

print("/nBest model:")
print(phase_best)


if not phase_best.exists():
    raise FileNotFoundError(
        f"Phase 1 best model was not found: {phase_best}"
    )

In [ ]:
print("\n" + "=" * 70)
print("FINAL TEST SET EVALUATION")
print("=" * 70)


best_model = YOLO(str(phase_best))


test_metrics = best_model.val(
    data=str(DATASET_DIR),

    split="test",

    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,

    project=PROJECT,
    name=TEST_NAME,

    plots=True
)


print("\n" + "=" * 70)
print("FINAL YOLO11 TEST RESULTS")
print("=" * 70)

print(
    f"Top-1 Test Accuracy: "
    f"{test_metrics.top1:.4f} "
    f"({test_metrics.top1 * 100:.2f}%)"
)

print(
    f"Top-5 Test Accuracy: "
    f"{test_metrics.top5:.4f} "
    f"({test_metrics.top5 * 100:.2f}%)"
)

In [ ]:
phase_csv = phase_dir / "results.csv"

df1 = pd.read_csv(phase_csv)


# Remove possible whitespace from column names
df1.columns = df1.columns.str.strip()


print("\nPhase columns:")
print(df1.columns.tolist())


In [ ]:
val_acc = (
    df1["metrics/accuracy_top1"].tolist()
)

train_loss = (
    df1["train/loss"].tolist()
)

val_loss = (
    df1["val/loss"].tolist()
)


epochs = range(1, len(val_acc) + 1)


In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    val_acc,
    label="Validation Top-1 Accuracy"
)


plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("YOLO11x Validation Accuracy - SSV")

plt.legend()

plt.xlim(left=0)

plt.tight_layout()

plt.show()


In [ ]:
%matplotlib inline
plt.figure(figsize=(10, 5))

plt.plot(
    epochs,
    train_loss,
    label="Train Loss"
)

plt.plot(
    epochs,
    val_loss,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("YOLO11x Loss - SSV")

plt.xlim(left=0)

plt.legend()

plt.tight_layout()

plt.show()